working directory

root:
  
  train

*   cancer
*   normal

  val


*   cancer
*   normal


    

In [ ]:
!mkdir -p /content/datasets/kau/train/birads1

In [ ]:
!mkdir -p /content/datasets/kau/val/birads1

In [1]:
import cv2
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import shutil
import os
import os.path as osp
from pathlib import Path
from PIL import Image
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [3]:
import kagglehub

In [ ]:
from imblearn.metrics import classification_report_imbalanced

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
from torchvision import datasets, models, transforms, ops
from torchvision.models.inception import InceptionOutputs

# Preprocessing

In [4]:
# Download latest version
path = kagglehub.dataset_download("orvile/kau-bcmd-mamography-dataset")

print("Path to dataset files:", path)

100%|██████████| 5.08G/5.08G [04:02<00:00, 22.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/orvile/kau-bcmd-mamography-dataset/versions/1


In [5]:
!ls /root/.cache/kagglehub/datasets/orvile/kau-bcmd-mamography-dataset/versions/1/King\ Abdulaziz\ University\ Mammogram\ Dataset/

birads1  birads3  birads4  birads5


In [6]:
paths = sorted(Path('/root/.cache/kagglehub/datasets/orvile/kau-bcmd-mamography-dataset/versions/1/King Abdulaziz University Mammogram Dataset').rglob('*/*.png'))
print(len(paths))

2378


In [11]:
# save image paths and label to list
list_images = []
list_labels = []
for path in paths:
    list_images.append(path)
    list_labels.append(str(path).split('/')[10])

In [ ]:
# split data for training and testing
X_train, X_test, y_train, y_test = train_test_split(list_images, list_labels, test_size=0.2, random_state=42, stratify=list_labels)

In [ ]:
# copy sample to working directory for faster computation
for x,y in zip(X_train,y_train):
    shutil.copyfile(x,f"/content/datasets/kau/train/{y}/{osp.basename(x)}")

In [ ]:
for x,y in zip(X_test,y_test):
    shutil.copyfile(x,f"/content/datasets/kau/val/{y}/{osp.basename(x)}")

In [ ]:
!rm -rf /content/datasets/kau/.ipynb_checkpoints/

In [ ]:
# compute class weights
cnt = np.unique(y_train, return_counts=True)[1]
avg = cnt/np.sum(cnt)
cw = 1/(np.log(1.02+avg))
cw = [float(c) for c in cw]
print(cw)

[1.694200610423119, 5.966676314041752, 16.339618667316845, 33.8425586909324]


# Data pipeline

In [ ]:
train_crop_size = 224
val_crop_size = 224
val_resize_size = 356

In [ ]:
# Data augmentation and normalization for training
# Just normalization for validation
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(train_crop_size),
        transforms.GaussianBlur(kernel_size=(3,3)),
        transforms.RandomEqualize(),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAdjustSharpness(sharpness_factor=2),
        transforms.RandomRotation(degrees=30),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.PILToTensor(),
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(val_resize_size),
        transforms.CenterCrop(val_crop_size),
        transforms.PILToTensor(),
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [ ]:
# load data
data_dir = '/content/datasets/kau'
batch_size = 64
dataloaders = {}
image_datasets = {x: datasets.ImageFolder(osp.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'val']}
# get class names
class_names = image_datasets['train'].classes
# combine class name and class weight
class_weights = dict(zip(class_names,cw))
# assign class weight to each train sample
samples_weight = torch.tensor([class_weights[t] for t in y_train])
# create weighted random sampler
sampler = data.WeightedRandomSampler(samples_weight, len(samples_weight), replacement=True)
# create data loader using the sampler
dataloaders['train'] = data.DataLoader(image_datasets['train'], batch_size=batch_size, sampler=sampler)
# dataloaders = {x: data.DataLoader(image_datasets[x], batch_size=64,
#                                   shuffle=True, num_workers=4)
#               for x in ['train', 'val']}

samples_weight = torch.tensor([class_weights[t] for t in y_test])
sampler = data.WeightedRandomSampler(samples_weight, len(samples_weight))
dataloaders['val'] = data.DataLoader(image_datasets['val'], batch_size=batch_size, sampler=sampler)

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}

# print(class_dict)

# Training process

In [ ]:
# visualization funciton to understand data augmentation
def imshow(inp, title=None):
    """Display image for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)  # pause a bit so that plots are updated

In [ ]:
def train_model(model, criterion, optimizer, scheduler, output_dir, num_epochs=25):
    since = time.time()

    # Create a temporary directory to save training checkpoints
    # with TemporaryDirectory() as tempdir:
    best_model_params_path = os.path.join(output_dir, 'best_model_params.pt')

    torch.save(model.state_dict(), best_model_params_path)
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0
            gt_labels = []
            pred_labels = []
            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)

                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                if phase == 'val':
                    # save ground truth and prediction labels for classification metrics
                    gt_labels.extend(labels.cpu().detach().numpy())
                    pred_labels.extend(preds.cpu().detach().numpy())

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), best_model_params_path)
            # break

        print()

    # print("Ground truth")
    # print(np.unique(gt_labels))
    # print(np.unique(pred_labels))
    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')
    print(classification_report_imbalanced(gt_labels, pred_labels, target_names=['cancer','normal'], zero_division=0))
    print(confusion_matrix(gt_labels, pred_labels))

    # load best model weights
    model.load_state_dict(torch.load(best_model_params_path, weights_only=True))

    return model

In [ ]:
def visualize_model(model, num_images=6):
    was_training = model.training
    model.eval()
    images_so_far = 0
    fig = plt.figure()

    with torch.no_grad():
        for i, (inputs, labels) in enumerate(dataloaders['val']):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for j in range(inputs.size()[0]):
                images_so_far += 1
                ax = plt.subplot(num_images//2, 2, images_so_far)
                ax.axis('off')
                ax.set_title(f'predicted: {class_names[preds[j]]}')
                imshow(inputs.cpu().data[j])

                if images_so_far == num_images:
                    model.train(mode=was_training)
                    return
        model.train(mode=was_training)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2, reduction='mean', ignore_index=-100):
        super(FocalLoss, self).__init__()
        self.alpha = alpha  # Class-wise weighting factor
        self.gamma = gamma  # Focusing parameter
        self.reduction = reduction
        self.ignore_index = ignore_index

    def forward(self, inputs, targets):
        # Calculate Cross Entropy Loss
        # F.cross_entropy expects raw logits as inputs
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', ignore_index=self.ignore_index)

        # Calculate probabilities from logits
        # pt represents the probability of the correct class
        pt = torch.exp(-ce_loss)

        # Calculate Focal Loss
        focal_loss = (self.alpha * (1 - pt)**self.gamma * ce_loss) if self.alpha is not None else ((1 - pt)**self.gamma * ce_loss)

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else 'cpu'

In [ ]:
# hyperparameters
num_classes = 4
learning_rate = 0.001
momentum = 0.9
weight_decay = 1e-4
lr_step_size = 7
lr_gamma = 0.1

In [ ]:
model_ft = models.resnet34(weights='IMAGENET1K_V1')
num_ftrs = model_ft.fc.in_features
# Here the size of each output sample is set to number of classes.
# Alternatively, it can be generalized to ``nn.Linear(num_ftrs, len(class_names))``.
model_ft.fc = nn.Linear(num_ftrs, num_classes)
# model_ft.classifier[1] = nn.Linear(in_features=1280, out_features=4)

model_ft = model_ft.to(device)

# define loss function
criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float), reduction='mean')
criterion = criterion.to(device)

# Observe that all parameters are being optimized
optimizer_ft = optim.SGD(model_ft.parameters(), lr=learning_rate, momentum=momentum)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = optim.lr_scheduler.StepLR(optimizer_ft, step_size=lr_step_size, gamma=lr_gamma)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 230MB/s]


In [ ]:
num_epochs = 20
output_dir = "/content/results/resnet34"
model_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler, output_dir, num_epochs=num_epochs)

Epoch 0/19
----------
train Loss: 1.3405 Acc: 0.5121
val Loss: 1.2460 Acc: 0.5819

Epoch 1/19
----------
train Loss: 1.1251 Acc: 0.6872
val Loss: 1.0399 Acc: 0.7647

Epoch 2/19
----------
train Loss: 1.1023 Acc: 0.6661
val Loss: 1.0723 Acc: 0.5840

Epoch 3/19
----------
train Loss: 0.9507 Acc: 0.7245
val Loss: 1.0779 Acc: 0.6113

Epoch 4/19
----------
train Loss: 0.9702 Acc: 0.7371
val Loss: 0.9191 Acc: 0.6891

Epoch 5/19
----------
train Loss: 0.9410 Acc: 0.7192
val Loss: 1.0329 Acc: 0.6197

Epoch 6/19
----------
train Loss: 0.9402 Acc: 0.7093
val Loss: 1.0684 Acc: 0.4706

Epoch 7/19
----------
train Loss: 0.9688 Acc: 0.6930
val Loss: 0.8567 Acc: 0.6660

Epoch 8/19
----------
train Loss: 0.9112 Acc: 0.7366
val Loss: 0.9964 Acc: 0.6134

Epoch 9/19
----------
train Loss: 0.9044 Acc: 0.7471
val Loss: 1.0381 Acc: 0.6975

Epoch 10/19
----------
train Loss: 0.9173 Acc: 0.7497
val Loss: 1.0732 Acc: 0.6660

Epoch 11/19
----------
train Loss: 0.8667 Acc: 0.7597
val Loss: 1.0773 Acc: 0.7206

Ep

In [ ]:
model_ft = models.resnet50(weights='IMAGENET1K_V1')
num_ftrs = model_ft.fc.in_features
# Here the size of each output sample is set to 2.
# Alternatively, it can be generalized to ``nn.Linear(num_ftrs, len(class_names))``.
model_ft.fc = nn.Linear(num_ftrs, 4)
# model_ft.classifier[1] = nn.Linear(in_features=1280, out_features=4)

model_ft = model_ft.to(device)

# define loss function
criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float), reduction='mean')
# criterion = FocalLoss()
criterion = criterion.to(device)
# Observe that all parameters are being optimized
optimizer_ft = optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = optim.lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

In [ ]:
num_epochs = 20
output_dir = "/content/results/resnet50"
model_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler, output_dir, num_epochs=num_epochs)

Epoch 0/19
----------
train Loss: 1.3266 Acc: 0.5931
val Loss: 1.2492 Acc: 0.7353

Epoch 1/19
----------
train Loss: 1.1772 Acc: 0.6961
val Loss: 1.1122 Acc: 0.7353

Epoch 2/19
----------
train Loss: 1.0292 Acc: 0.7355
val Loss: 1.1006 Acc: 0.7899

Epoch 3/19
----------
train Loss: 1.0113 Acc: 0.7261
val Loss: 1.1550 Acc: 0.6429

Epoch 4/19
----------
train Loss: 0.9787 Acc: 0.7287
val Loss: 1.1111 Acc: 0.4832

Epoch 5/19
----------
train Loss: 0.9560 Acc: 0.6961
val Loss: 1.1001 Acc: 0.6660

Epoch 6/19
----------
train Loss: 0.8935 Acc: 0.7466
val Loss: 1.1075 Acc: 0.4076

Epoch 7/19
----------
train Loss: 0.9096 Acc: 0.7056
val Loss: 0.9461 Acc: 0.6975

Epoch 8/19
----------
train Loss: 0.8324 Acc: 0.7471
val Loss: 1.0554 Acc: 0.6597

Epoch 9/19
----------
train Loss: 0.8616 Acc: 0.7518
val Loss: 0.8934 Acc: 0.6429

Epoch 10/19
----------
train Loss: 0.8950 Acc: 0.7476
val Loss: 0.9252 Acc: 0.6702

Epoch 11/19
----------
train Loss: 0.8450 Acc: 0.7613
val Loss: 0.9065 Acc: 0.5756

Ep

In [ ]:
model_ft = models.efficientnet_b0(weights='IMAGENET1K_V1')
# num_ftrs = model_ft.fc.in_features
# num_ftrs = model_ft.classifier.in_features
# Here the size of each output sample is set to 2.
# Alternatively, it can be generalized to ``nn.Linear(num_ftrs, len(class_names))``.
# model_ft.fc = nn.Linear(num_ftrs, 4)
# model_ft.classifier = nn.Linear(in_features=1024, out_features=4)
model_ft.classifier[1] = nn.Linear(in_features=1280, out_features=4)

model_ft = model_ft.to(device)

# define loss function
criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float))
criterion = criterion.to(device)
# Observe that all parameters are being optimized
optimizer_ft = optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = optim.lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 190MB/s]


In [ ]:
num_epochs = 20
output_dir = "/content/results/efficientnet_b0"
model_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler, output_dir, num_epochs=num_epochs)

Epoch 0/19
----------
train Loss: 1.3415 Acc: 0.4690
val Loss: 1.3248 Acc: 0.6891

Epoch 1/19
----------
train Loss: 1.2831 Acc: 0.6688
val Loss: 1.3158 Acc: 0.6702

Epoch 2/19
----------
train Loss: 1.2149 Acc: 0.6656
val Loss: 1.1679 Acc: 0.6639

Epoch 3/19
----------
train Loss: 1.2100 Acc: 0.6504
val Loss: 1.2354 Acc: 0.6681

Epoch 4/19
----------
train Loss: 1.1652 Acc: 0.6777
val Loss: 1.1385 Acc: 0.6555

Epoch 5/19
----------
train Loss: 1.1437 Acc: 0.7008
val Loss: 1.0602 Acc: 0.7185

Epoch 6/19
----------
train Loss: 1.1465 Acc: 0.6556
val Loss: 1.2436 Acc: 0.4643

Epoch 7/19
----------
train Loss: 1.1171 Acc: 0.6809
val Loss: 1.0136 Acc: 0.7395

Epoch 8/19
----------
train Loss: 1.1338 Acc: 0.6772
val Loss: 1.0915 Acc: 0.7164

Epoch 9/19
----------
train Loss: 1.0958 Acc: 0.6861
val Loss: 1.0185 Acc: 0.7647

Epoch 10/19
----------
train Loss: 1.0805 Acc: 0.6972
val Loss: 0.9610 Acc: 0.7983

Epoch 11/19
----------
train Loss: 1.0872 Acc: 0.6824
val Loss: 0.9151 Acc: 0.7647

Ep